Evaluate state of trained model and statistics generated from `scripts/evaluate` on the validation fold (0)

In [14]:
config_dir = "/Users/sophiali/Desktop/chip-stroma-analysis/configs"
version = "v6"
results_dir = "/Users/sophiali/Desktop/chip-stroma-analysis/results"
single_model = False

In [2]:
import sys
import os

from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root / "src"))

os.chdir(Path.cwd().parent)

import pandas as pd
import argparse as ap
import numpy as np

from pathlib import Path

from chip_stroma.utils.header_footers import log_header, log_footer
from chip_stroma.utils.config import load_configs
from chip_stroma.utils.loggers import setup_logger
from chip_stroma.utils.io import load_all_fold_patch_metrics

from chip_stroma.evaluate.segmentation_stats import (
    per_fold_metrics,
    optuna_importance,
    threshold_sweep,
    select_overlay_cases,
    top_k_trials_table,
    multiseed_summary_table,
    final_cv_summary_table
)

logger = setup_logger(__name__)

%load_ext autoreload
%autoreload 2

In [25]:
# Load workflow and path configurations
config = load_configs(
    pipeline = Path(config_dir) / "08_evaluate.yaml",
    paths    = Path(config_dir) / "00_paths.yaml"
)
n_folds = 2

2026-07-29 16:24:51,753 | INFO     | chip_stroma.utils.config       | ==================================================
2026-07-29 16:24:51,754 | INFO     | chip_stroma.utils.config       | Step 01: Configurations
2026-07-29 16:24:51,755 | INFO     | chip_stroma.utils.config       | - Pipeline : configs/08_evaluate.yaml
2026-07-29 16:24:51,755 | INFO     | chip_stroma.utils.config       | - Paths    : configs/00_paths.yaml
2026-07-29 16:24:51,755 | INFO     | chip_stroma.utils.config       | --------------------------------------------------
2026-07-29 16:24:51,760 | INFO     | chip_stroma.utils.config       | Successfully loaded and merged both configuration files
2026-07-29 16:24:51,760 | INFO     | chip_stroma.utils.config       | ==================================================


In [13]:
# Initialize version results directory
dst_dir = Path(results_dir) / version
inference_dir = dst_dir / "inference"
evaluate_dir  = dst_dir / "evaluate"
evaluate_dir.mkdir(parents = True, exist_ok = True)

In [26]:
# 1. Compute per-patient segmentation metrics, per fold
predictions = load_all_fold_patch_metrics(
    src_dir      = inference_dir,
    n_folds      = n_folds,
    single_model = single_model
)

In [27]:
fold_metrics = per_fold_metrics(predictions)
fold_metrics.to_csv(evaluate_dir / "per_fold_metrics.csv", index = False)

In [ ]:
# 2. Compute a threshold sweep, pooled across folds
probs =pd.concat([pd.read_pickle(inference_dir/f"fold_{f}"/"val_probs.pkl") 
                    for f in range(n_folds)])
gt    = pd.concat([pd.read_pickle(inference_dir / f"fold_{f}"/"val_gt.pkl") 
                    for f in range(n_folds)])